# Step 1 — immutable Kaggle T4 x2 dense seed-0 run

The runner executes its diagnostic preflight under `diagnostic-preflight/` and its dense baseline only under `production/`. The post-run checks below enforce operational completion exactly; they print scientific measurements without imposing a capability threshold.

This checked-in file is a template. After committing and pushing the runner, render a pinned copy with `python step1/kaggle/render_preflight_notebook.py --notebook step1/kaggle/step1_t4x2.ipynb --commit <final-40-char-sha>`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/escher-bach/actuallybuildingstuff.git'
GIT_COMMIT = '__FINAL_COMMIT_SHA__'
CONFIG_REL = 'step1/configs/kaggle/t4x2_dense_seed0.toml'

if not (len(GIT_COMMIT) == 40 and all(c in '0123456789abcdef' for c in GIT_COMMIT)):
    raise RuntimeError('Generate this template with render_preflight_notebook.py after committing the runner.')
WORKING = Path('/kaggle/working')
SOURCE = WORKING / 'actuallybuildingstuff'
PROJECT = SOURCE / 'baby-llm-foundations'
OUTPUT = WORKING / 'step1-results'
assert not SOURCE.exists(), f'fresh batch session required; already exists: {SOURCE}'
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
env = os.environ.copy()
env.update({'GIT_TERMINAL_PROMPT': '0', 'PYTHONUNBUFFERED': '1', 'PIP_DISABLE_PIP_VERSION_CHECK': '1', 'WANDB_MODE': 'disabled', 'TOKENIZERS_PARALLELISM': 'false'})
subprocess.run(['git', 'clone', REPO_URL, str(SOURCE)], check=True, env=env)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', GIT_COMMIT], check=True, env=env)
resolved = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True, env=env).strip()
assert resolved == GIT_COMMIT, (resolved, GIT_COMMIT)
assert (PROJECT / CONFIG_REL).is_file(), PROJECT / CONFIG_REL


In [ ]:
cmd = [sys.executable, '-m', 'step1_experiments.runner', '--config', str(PROJECT / CONFIG_REL), '--output-root', str(OUTPUT), '--resume', 'auto']
completed = subprocess.run(cmd, cwd=str(PROJECT / 'step1' / 'python'), env=env, check=False)
if completed.returncode != 0:
    raise RuntimeError(f'Step 1 runner failed with exit code {completed.returncode}; download the failure bundle from {OUTPUT}')


In [ ]:
import hashlib
import json
import math
import tomllib

reports = sorted(OUTPUT.glob('*/production/training_report.json'))
assert len(reports) == 1, reports
report_path = reports[0]
run_dir = report_path.parents[1]
production = run_dir / 'production'
report = json.loads(report_path.read_text())
config = tomllib.loads((PROJECT / CONFIG_REL).read_text())
config_hash = hashlib.sha256(json.dumps(config, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
training, world = config['training'], config['world']
world_size = 2
microstep_tokens = world_size * training['microbatch_sequences'] * world['context_length']
assert training['global_tokens_per_update'] % microstep_tokens == 0, training
assert training['token_budget'] % training['global_tokens_per_update'] == 0, training
expected_steps = training['token_budget'] // training['global_tokens_per_update']
expected_plan = {
    'world_size': world_size,
    'per_device_sequences': training['microbatch_sequences'],
    'context_length': world['context_length'],
    'nominal_global_input_tokens_per_microstep': microstep_tokens,
    'gradient_accumulation_steps': training['global_tokens_per_update'] // microstep_tokens,
    'nominal_global_input_tokens_per_update': training['global_tokens_per_update'],
    'token_budget': training['token_budget'],
    'max_steps': expected_steps,
    'checkpoint_interval_updates': training['checkpoint_interval_updates'],
    'checkpoint_interval_nominal_global_input_tokens': training['checkpoint_interval_updates'] * training['global_tokens_per_update'],
    'checkpoint_total_limit': training['checkpoint_total_limit'],
    'token_unit': 'nominal global input tokens (world_size * per_device_sequences * context_length)',
    'supervised_token_unit': 'action-label tokens after the causal shift; variable and reported separately',
}
expected_checkpoint = production / 'checkpoints' / f'checkpoint-{expected_steps}'
expected_model = production / 'model'
assert report['contract'] == 'step1_dense_training_v1', report
assert report['global_step'] == expected_steps, report
assert Path(report['last_trainer_checkpoint']) == expected_checkpoint.resolve(), report
assert expected_checkpoint.is_dir(), expected_checkpoint
assert Path(report['model_artifact']) == expected_model.resolve(), report
assert (expected_model / 'config.json').is_file() and (expected_model / 'experiment.json').is_file(), expected_model
assert report['ranks_finished'] == [0, 1], report
assert report['config_hash'] == config_hash and report['source_git_sha'] == GIT_COMMIT, report
assert report['token_accounting'] == expected_plan, report
assert report['serialization']['exact_state_dict']['exact'] is True, report
assert report['serialization']['exact_state_dict']['expected_state_sha256'] == report['serialization']['exact_state_dict']['actual_state_sha256'], report
metrics_path = run_dir / 'evaluation' / 'metrics.json'
metrics = json.loads(metrics_path.read_text())
expected_sets = {'validation', 'structural', 'rendering_b', 'reversible_control'}
expected_metrics = {'success_rate', 'failure_rate', 'malformed_action_rate', 'invalid_action_rate', 'mean_spent', 'mean_success_excess_cost', 'mean_steps'}
assert set(metrics['sets']) == expected_sets, metrics
assert math.isfinite(metrics['teacher_forced_action_nll']), metrics
for item in metrics['sets'].values():
    values = item['metrics']
    assert item['comparison']['label'] and item['comparison']['seed_policy'], item
    assert set(values) == expected_metrics, values
    assert all(value is None or math.isfinite(value) for value in values.values()), values
print(json.dumps(metrics, indent=2, sort_keys=True))
